# Notebook 04c: Honest evaluation, your own photos, and the live demo (Phase 4, step c)

**Goal:** look at what the models from 04b can *really* do, and let you try them yourself.
1. **Report** the group-split test results for every model, with the confusion matrix, per-chord scores, and the coverage-vs-accuracy trade-off.
2. **Error gallery:** wrong predictions drawn with the Phase 1-3 overlay, so each mistake can be traced to a cause (a missed hand, a wrong neck, a genuinely ambiguous chord).
3. **Your own photos** (optional): the number that tells you whether this works for *your* guitar and camera. Models never saw these photos.
4. **Live demo** with your webcam: chord prediction on the live video, with smoothing and a "not sure" answer.

**Not in this notebook:** more training or any accuracy tuning (that is the next phase).

### How a photo becomes a chord (the whole pipeline in one picture)
```
photo --> Phase 1 MediaPipe --> fingertips ----+
      \-> Phase 2 OpenCV    --> neck lines     +--> Phase 3 --> fingertip (string, fret) numbers --> Model B (trees) -+
                                                                                                                    +--> blend --> chord, or "not sure"
photo --> crop around neck + hand (from the same boxes) --> Model A2 (CNN) ----------------------------------------+
```
The blend trusts Model B only as much as Phase 3 was confident, and uses the CNN alone when no fingertip features exist.

### Needs
`p4_models.zip` from notebook 04b (found automatically if 04b ran in this same Colab session, otherwise a file picker asks for it). A GPU runtime is recommended for the CNNs, but inference also works on CPU.

## 0. Setup
**What:** installs `mediapipe` and `datasets` if missing, and prepares folders. No Google Drive.

In [ ]:
# colab-only
import os
PROJECT_DIR = '/content/Guitar-Chord-Recognition'
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
print('Project folder:', PROJECT_DIR)

In [ ]:
import os, sys, subprocess, importlib
def _ensure(pkg, mod=None):
    try:
        importlib.import_module(mod or pkg)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', pkg], check=True)
        importlib.invalidate_caches()
_ensure('mediapipe'); _ensure('datasets')
import mediapipe as mp
import torch
print('mediapipe', mp.__version__, '| torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU is fine for inference)')

## 1. Load the trained models and the Phase 1-3 code
**What:** loads the artifacts from 04b (`A1_group.pt`, `A2_crop.pt`, `model_b.joblib`, `fusion.json`, `predictions.npz`), then the same Phase 1-3 function code used in 04a (copied unchanged: notebook 03's mapping, which copies notebook 02's detector).
**Why the same code:** the model was trained on features produced by exactly this code. Using a different version at test time would silently shift the numbers.
**Note:** if 04b ran as a *rehearsal* (small smoke data), `fusion.json` says so and everything below is a pipeline check, not a result.

In [ ]:
import json, time, shutil, zipfile, collections, base64, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

P4 = Path(PROJECT_DIR) / 'outputs' / 'phase4'
MODELS = P4 / 'models'
if not (MODELS / 'fusion.json').exists():
    from google.colab import files
    print('Choose p4_models.zip (made by notebook 04b) ...')
    up = files.upload()
    MODELS.mkdir(parents=True, exist_ok=True)
    zp = Path(PROJECT_DIR) / 'p4_models.zip'
    zp.write_bytes(up[[n for n in up if n.lower().endswith('.zip')][0]])
    with zipfile.ZipFile(zp) as z:
        z.extractall(MODELS)
SET = json.loads((MODELS / 'fusion.json').read_text(encoding='utf-8'))
NAMES, BASE, ALPHA, THRESH, B_COLS = SET['classes'], SET['base_cnn'], SET['alpha'], SET['abstain_threshold'], SET['b_columns']
print('chords:', NAMES)
print(f"blend CNN: {BASE} | alpha {ALPHA} | abstain below {THRESH} | trained as {'REHEARSAL (numbers are not meaningful)' if SET['rehearsal'] else 'FULL run'}")

In [ ]:
# Phase 1/3 settings and imports: copied from notebooks/03_fingertip_fret_mapping.ipynb (fingerprint 1042900f2f). Do not edit here.
import json, glob, random, zipfile, urllib.request
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MAPCFG = dict(
    string_margin_min=0.30,   # 0 = exactly halfway between two strings, 1 = exactly on one string
    fret_margin_min=0.12,     # fraction of the fret-region width that the tip must stay away from a fret line
    outside_frac=0.6,         # beyond the outer strings by more than this x a string gap = not on the neck
    near_neck_frac=0.5,       # when choosing the fretting hand, tips this close (x string gap) to the neck count as near
    accept_conf=0.35,         # minimum overall confidence for 'accepted'
    geometry_quality={'high': 1.0, 'low': 0.6},   # how much we trust Phase 2's own confidence label
    allow_low_geometry=False, # False: low-confidence Phase 2 geometry gives 'tentative' results only
    fret_ratio=2 ** (-1 / 12),# real fret spacing shrinks by this factor per fret (12-tone equal temperament)
    wide_region_ratio=1.8,    # a fret region this many times wider than its neighbours hides a missed fret line
    narrow_region_ratio=0.45, # ... and this many times narrower suggests a spurious line
)

from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

MODEL_URL = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task'
MODEL_PATH = f'{PROJECT_DIR}/hand_landmarker.task'
if not os.path.exists(MODEL_PATH):
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)       # ~8 MB pretrained model from Google

FINGERTIPS = {4: 'Thumb', 8: 'Index', 12: 'Middle', 16: 'Ring', 20: 'Pinky'}
FINGER_COLORS = {'Thumb': (0, 165, 255), 'Index': (0, 255, 0), 'Middle': (255, 128, 0), 'Ring': (255, 0, 255), 'Pinky': (0, 255, 255)}   # BGR, as in Phase 1
CONNECTIONS = [(0, 1), (1, 2), (2, 3), (3, 4), (0, 5), (5, 6), (6, 7), (7, 8), (5, 9), (9, 10), (10, 11),
               (11, 12), (9, 13), (13, 14), (14, 15), (15, 16), (13, 17), (17, 18), (18, 19), (19, 20), (0, 17)]

def detect_hands(bgr, image_id):
    # Returns the Phase 1 export for one image. All coordinates are ORIGINAL-image pixels (floats).
    h, w = bgr.shape[:2]
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)               # MediaPipe wants RGB, OpenCV gives BGR
    res = LANDMARKER.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
    hands = []
    for i, lms in enumerate(res.hand_landmarks):
        cat = res.handedness[i][0]
        pts = [[round(lm.x * w, 2), round(lm.y * h, 2), round(lm.z, 4)] for lm in lms]   # normalized -> pixels
        tips = {name: dict(id=idx, x_px=pts[idx][0], y_px=pts[idx][1], z=pts[idx][2]) for idx, name in FINGERTIPS.items()}
        hands.append(dict(handedness_as_model_sees_it=cat.category_name, score=round(float(cat.score), 3),
                          landmarks_px=pts, tips=tips))
    return dict(schema_version=1, image=dict(id=image_id, width=w, height=h), hands=hands)


def check_coordinates(image_id, bgr, hands, p2):
    # Requirement 4: all three sources must describe the same pixel grid. Returns a list of problems (empty = fine).
    h, w = bgr.shape[:2]
    problems = []
    if (hands['image']['width'], hands['image']['height']) != (w, h):
        problems.append('Phase 1 export size differs from the image')
    if (p2['image']['width'], p2['image']['height']) != (w, h):
        problems.append(f"Phase 2 size {p2['image']['width']}x{p2['image']['height']} differs from the image {w}x{h}")
    if p2.get('image', {}).get('name') != image_id:
        problems.append('Phase 2 image name differs from the image ID')
    if p2.get('fretboard'):
        poly = np.array(p2['fretboard']['polygon'])
        if poly.min() < -2 * max(w, h) or poly.max() > 3 * max(w, h):
            problems.append('Phase 2 polygon is wildly outside the image (wrong coordinate frame?)')
    return problems

In [ ]:
# Phase 2 detector: copied from notebook 03 / notebook 02 (fingerprint 1042900f2f). Do not edit here.
# Phase 2 detector: copied from notebooks/02_fretboard_detection.ipynb (fingerprint 3bc032056c). Do not edit here.
import os, json, glob, random
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CFG = dict(
    max_side=800,            # resize so the longest image side is at most this many pixels (speed + consistent scale)
    canny_sigma=0.33,        # Canny thresholds are set from the image's median brightness (used for the fret lines)
    hough_threshold=40,      # votes a line needs in the Hough transform (higher = fewer, surer lines)
    hough_min_len_frac=0.08, # shortest accepted segment, as a fraction of the longest image side
    hough_max_gap=8,         # gaps (px) in an edge line that are bridged
    ridge_sigmas=(1.2, 2.0), # line widths (px) the ridge filter looks for; thin strings, slightly thicker fret wires
    angle_step=2.0,         # the neck-direction search tries every this-many degrees
    n_angle_candidates=4,    # how many different neck directions are fully tested; the best-scoring one wins
    string_gap_min_frac=0.015,  # smallest allowed spacing between neighbouring strings (x longest image side)
    string_gap_max_frac=0.09,   # largest allowed spacing between neighbouring strings
    string_min_extent_frac=0.2, # strings must run along at least this fraction of the image (rejects small noisy patches)
    string_min_density=0.5,     # at least this fraction of each string's length must show real line evidence
    fret_max_tilt=30,        # frets must be within this many degrees of perpendicular to the strings
    fret_min_cover=0.5,      # a fret must span at least this fraction of the neck width
)

# Where outputs go. PROJECT_DIR is defined in cell 0.


def show_images(images, titles, cols=3, size=4):
    # Display a list of images in a grid. Grayscale images (2D) are shown in gray.
    rows = int(np.ceil(len(images) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * size, rows * size))
    axes = np.atleast_1d(axes).ravel()
    for ax, im, t in zip(axes, images, titles):
        if im.ndim == 2:
            ax.imshow(im, cmap='gray')
        else:
            ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))   # OpenCV is BGR, matplotlib is RGB
        ax.set_title(t, fontsize=9); ax.axis('off')
    for ax in axes[len(images):]:
        ax.axis('off')
    plt.tight_layout(); plt.show()


def ridge_response(gray, valid):
    # Highlights thin bright or dark lines (strings, frets) as one response each, scaled to 0..1.
    g = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray).astype(np.float32) / 255.0   # local contrast boost
    best = np.zeros_like(g)
    for s in CFG['ridge_sigmas']:                                    # two line widths; keep the stronger response
        gs = cv2.GaussianBlur(g, (0, 0), s)
        dxx = cv2.Sobel(gs, cv2.CV_32F, 2, 0, ksize=3)               # second derivatives = how sharply brightness bends
        dyy = cv2.Sobel(gs, cv2.CV_32F, 0, 2, ksize=3)
        dxy = cv2.Sobel(gs, cv2.CV_32F, 1, 1, ksize=3)
        t = np.sqrt(((dxx - dyy) / 2) ** 2 + dxy ** 2)
        l1 = (dxx + dyy) / 2 + t                                     # the two eigenvalues of the Hessian (l1 >= l2)
        l2 = (dxx + dyy) / 2 - t
        bright = np.maximum(np.where(np.abs(l1) < 0.5 * np.abs(l2), -l2, 0), 0)   # thin BRIGHT line: one strongly negative curvature
        dark = np.maximum(np.where(np.abs(l2) < 0.5 * np.abs(l1), l1, 0), 0)      # thin DARK line: one strongly positive curvature
        best = np.maximum(best, (bright + dark) * s ** 2)            # s**2 makes the two scales comparable
    best *= (valid > 0)
    ref = np.percentile(best[valid > 0], 97) if (valid > 0).any() else 1.0
    return np.clip(best / max(ref, 1e-6), 0, 1).astype(np.float32)   # clipping stops one huge edge from dominating


def preprocess(bgr, max_side=800):
    h0, w0 = bgr.shape[:2]
    scale = min(1.0, max_side / max(h0, w0))                       # never enlarge, only shrink
    img = cv2.resize(bgr, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA) if scale < 1 else bgr.copy()

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)                        # 5x5 kernel, sigma chosen automatically

    med = float(np.median(blur))
    lo = int(max(20, (1 - CFG['canny_sigma']) * med))
    hi = int(max(60, (1 + CFG['canny_sigma']) * med))
    edges_raw = cv2.Canny(blur, lo, hi)

    # Mask out the black padding that rotation-augmented images have, and the edge where it meets the photo
    valid = (gray > 8).astype(np.uint8)
    valid = cv2.morphologyEx(valid, cv2.MORPH_OPEN, np.ones((9, 9), np.uint8))
    valid = cv2.erode(valid, np.ones((13, 13), np.uint8))
    edges = cv2.bitwise_and(edges_raw, edges_raw, mask=valid)

    ridge = ridge_response(gray, valid)
    return dict(img=img, gray=gray, blur=blur, edges_raw=edges_raw, edges=edges, ridge=ridge, scale=scale, canny=(lo, hi))


def seg_lengths(segs):
    return np.hypot(segs[:, 2] - segs[:, 0], segs[:, 3] - segs[:, 1])

def seg_angles(segs):
    # direction of each segment in degrees, in [0, 180). 0 = pointing right, 90 = pointing down.
    return np.degrees(np.arctan2(segs[:, 3] - segs[:, 1], segs[:, 2] - segs[:, 0])) % 180.0

def signed_angle(deg):
    # wraps angles to [-90, 90) so "almost horizontal" is near 0 whichever way the line points
    return (deg + 90.0) % 180.0 - 90.0


def hough_segments(edges):
    h, w = edges.shape
    lines = cv2.HoughLinesP(edges, rho=1, theta=np.pi / 180, threshold=CFG['hough_threshold'],
                            minLineLength=int(CFG['hough_min_len_frac'] * max(h, w)),
                            maxLineGap=CFG['hough_max_gap'])
    if lines is None:
        return np.zeros((0, 4))
    return lines.reshape(-1, 4).astype(float)     # shape differs between OpenCV versions; this handles both


def neck_matrix(angle, w, h):
    # Rotation that makes lines pointing at `angle` degrees horizontal. The output canvas is the image diagonal wide,
    # so nothing is cropped when the photo is rotated. Returns the 2x3 matrix and the canvas size.
    D = int(np.ceil(np.hypot(w, h)))
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    M[0, 2] += D / 2 - w / 2
    M[1, 2] += D / 2 - h / 2
    return M, D


def to_neck_frame(segs, M):
    # Apply the same rotation to segment endpoints.
    pts = segs.reshape(-1, 2) @ M[:, :2].T + M[:, 2]
    return pts.reshape(-1, 4)


def comb_search(profile, d_min, d_max, n=6, tol=2):
    # profile: ridge evidence per row. Find the best place and spacing d for n equally spaced teeth.
    p = np.convolve(profile, np.ones(3) / 3, mode='same')
    base = np.convolve(p, np.ones(41) / 41, mode='same')                  # slowly varying background level
    p = np.maximum(p - base, 0).astype(np.float32)                         # keep only peaks standing above it
    pd_ = cv2.dilate(p.reshape(-1, 1), np.ones((2 * tol + 1, 1), np.uint8)).ravel()   # each tooth may be +-tol rows off
    best = (0.0, 0, 0)                                                     # (score, first row, spacing)
    for d in range(d_min, d_max + 1):
        offs = np.arange(n) * d
        starts = np.arange(0, len(pd_) - offs[-1])
        if len(starts) == 0:
            break
        v = np.sort(pd_[starts[:, None] + offs[None, :]], axis=1)          # evidence under each tooth, weakest first
        # drop the weakest tooth (a string may be hidden) but reward teeth of similar strength
        score = v[:, 1:].sum(1) * np.sqrt(v[:, 1] / (v[:, -1] + 1e-9))
        i = int(np.argmax(score))
        if score[i] > best[0]:
            best = (float(score[i]), int(starts[i]), d)
    return best


def neck_candidates(ridge, n=None):
    # Returns the best few distinct neck directions as dicts, and (angles, scores) for plotting.
    n = n or CFG['n_angle_candidates']
    h, w = ridge.shape
    big = max(h, w)
    d_min, d_max = max(8, int(CFG['string_gap_min_frac'] * big)), int(CFG['string_gap_max_frac'] * big)
    phis = np.arange(0, 180, CFG['angle_step'])
    scores, info = np.zeros(len(phis)), []
    for i, phi in enumerate(phis):
        M, D = neck_matrix(phi, w, h)
        rot = cv2.warpAffine(ridge, M, (D, D))                             # ridge map with direction phi made horizontal
        coherent = np.maximum(cv2.blur(rot, (41, 1)) - 0.35, 0)            # keep only long unbroken lines
        sc, y_start, d = comb_search(coherent.sum(axis=1), d_min, d_max)
        scores[i] = sc
        info.append((y_start, d))
    picked = []
    for i in np.argsort(scores)[::-1]:                                     # best first; skip directions too close to a pick
        if scores[i] <= 0:
            break
        if all(min(abs(phis[i] - phis[j]), 180 - abs(phis[i] - phis[j])) >= 10 for j in picked):
            picked.append(int(i))
        if len(picked) == n:
            break
    cands = [dict(angle=float(phis[i]), comb_score=float(scores[i]), y_start=info[i][0], d=info[i][1]) for i in picked]
    return cands, (phis, scores)


def string_y(st, x):
    # y position of a string at horizontal position x (strings are lines y = y0 + m*(x - xc))
    return st['y0'] + st['m'] * (x - st['xc'])


def _runs(mask):
    # (start, end) of each run of True values in a 1-D boolean array
    edges = np.diff(np.r_[0, mask.astype(np.int8), 0])
    return list(zip(np.nonzero(edges == 1)[0], np.nonzero(edges == -1)[0] - 1))


def find_strings(ridge, cand, n_strings=6):
    # Refine the comb's rough guess into 6 fitted string lines in the neck-aligned view. Returns a dict, or None.
    h, w = ridge.shape
    M, D = neck_matrix(cand['angle'], w, h)
    rot = cv2.warpAffine(ridge, M, (D, D))
    ys, xs = np.nonzero(rot > 0.25)                                        # ridge pixels worth fitting to
    wt = rot[ys, xs].astype(float); ys = ys.astype(float); xs = xs.astype(float)
    d = cand['d']
    y_init = [cand['y_start'] + i * d for i in range(n_strings)]
    near = np.zeros(len(ys), bool)
    for yi in y_init:
        near |= np.abs(ys - yi) <= 0.35 * d
    if near.sum() < 50:
        return None
    xc = float(np.average(xs[near], weights=wt[near]))                     # shared pivot column in the middle of the evidence

    # 1. per-string slope search (perspective makes strings fan out)
    slopes = []
    for yi in y_init:
        scores = [(wt[np.abs(ys - (yi + t * (xs - xc))) <= 2.0].sum(), t) for t in np.tan(np.radians(np.arange(-12, 12.1, 1.0)))]
        slopes.append(max(scores)[1])
    # 2. slopes should vary smoothly across the 6 strings: fit a line through them and stay close to it
    idx = np.arange(n_strings)
    a, b = np.polyfit(idx, slopes, 1)
    model = a * idx + b
    lines = []
    for i, yi in enumerate(y_init):
        y0, m = yi, float(model[i])
        # 3. weighted least squares on nearby ridge pixels, with a narrowing band
        for hw in (0.30 * d, 0.22 * d, 0.16 * d):
            sel = np.abs(ys - (y0 + m * (xs - xc))) <= hw
            if sel.sum() < 25:
                break
            A = np.polyfit(xs[sel] - xc, ys[sel], 1, w=wt[sel])
            m = float(np.clip(A[0], model[i] - 0.04, model[i] + 0.04))     # at most ~2.3 degrees away from the fan model
            y0 = float(A[1])
        lines.append((y0, m))

    # 4. where along the neck is there real evidence? (a string may be hidden by fingers over part of its length)
    X = np.arange(D)
    support, count = [], np.zeros(D)
    for y0, m in lines:
        yy = np.rint(y0 + m * (X - xc)).astype(int)
        ok = (yy >= 1) & (yy < D - 1)
        v = np.zeros(D)
        for dy in (-1, 0, 1):
            v[ok] = np.maximum(v[ok], rot[yy[ok] + dy, X[ok]])
        s = cv2.morphologyEx((v > 0.2).astype(np.uint8).reshape(1, -1), cv2.MORPH_CLOSE, np.ones((1, 31), np.uint8)).ravel()
        support.append(s); count += s
    good = cv2.morphologyEx((count >= 4).astype(np.uint8).reshape(1, -1), cv2.MORPH_CLOSE, np.ones((1, 41), np.uint8)).ravel()
    runs = _runs(good > 0)
    if not runs:
        return None
    xl, xr = max(runs, key=lambda r: count[r[0]:r[1] + 1].sum())
    strings = []
    for (y0, m), s in zip(lines, support):
        dens = float(s[xl:xr + 1].mean())
        strings.append(dict(xc=xc, y0=y0, m=m, x1=float(xl), x2=float(xr), length=float((xr - xl) * dens), density=dens))

    # 5. sanity checks: strings must stay in order (no crossing) over the whole stretch, with fairly even spacing
    gaps_l = np.diff([string_y(s, xl) for s in strings]); gaps_r = np.diff([string_y(s, xr) for s in strings])
    if gaps_l.min() < 2 or gaps_r.min() < 2:
        return None
    gaps = np.diff([s['y0'] for s in strings])
    cv = float(gaps.std() / gaps.mean())
    return dict(M=M, strings=strings, cv=cv, xl=float(xl), xr=float(xr), extent=float(xr - xl),
                density=float(np.mean([s['density'] for s in strings])))


def merge_intervals(iv):
    # total length covered by a set of (start, end) intervals, counting overlaps once
    iv = sorted(iv); total, cur_s, cur_e = 0.0, None, None
    for a, b in iv:
        if cur_e is None or a > cur_e:
            if cur_e is not None:
                total += cur_e - cur_s
            cur_s, cur_e = a, b
        else:
            cur_e = max(cur_e, b)
    if cur_e is not None:
        total += cur_e - cur_s
    return total


def find_frets(sr, strings, spacing):
    # strings: the 6 chosen strings. spacing: mean distance between neighboring strings.
    xc = strings[0]['xc']
    y_top = strings[0]['y0'] - spacing / 2          # neck band edges at x = xc
    y_bot = strings[-1]['y0'] + spacing / 2
    band = y_bot - y_top
    y_mid = (y_top + y_bot) / 2

    ang = np.abs(signed_angle(seg_angles(sr)))
    s = sr[ang >= 90 - CFG['fret_max_tilt']].copy()                       # keep almost-vertical segments
    if len(s) == 0:
        return [], dict(y_top=y_top, y_bot=y_bot, y_mid=y_mid)
    swap = s[:, 1] > s[:, 3]                                              # make every segment point downward
    s[swap] = s[swap][:, [2, 3, 0, 1]]

    items = []
    for x1, y1, x2, y2 in s:
        if y2 - y1 < 1e-6:
            continue
        a, b = max(y1, y_top), min(y2, y_bot)                              # part of the segment inside the neck band
        if b - a < 0.25 * band:
            continue
        k = (x2 - x1) / (y2 - y1)                                          # horizontal shift per unit of y
        items.append((x1 + k * (y_mid - y1), k, a, b, y2 - y1))            # (x at band middle, k, y-range, full length)
    if not items:
        return [], dict(y_top=y_top, y_bot=y_bot, y_mid=y_mid)

    items.sort(key=lambda t: t[0])
    tol = max(3.0, 0.8 * spacing)
    groups, cur = [], [items[0]]
    for it in items[1:]:
        if it[0] - cur[-1][0] <= tol:
            cur.append(it)
        else:
            groups.append(cur); cur = [it]
    groups.append(cur)

    frets = []
    for g in groups:
        cover = merge_intervals([(a, b) for _, _, a, b, _ in g]) / band
        if cover < CFG['fret_min_cover']:
            continue
        wgt = np.array([b - a for _, _, a, b, _ in g])
        frets.append(dict(xm=float(np.average([t[0] for t in g], weights=wgt)),
                          k=float(np.average([t[1] for t in g], weights=wgt)), cover=float(cover),
                          span=float(max(t[4] for t in g))))             # longest piece seen = real neck width
    return frets, dict(y_top=y_top, y_bot=y_bot, y_mid=y_mid)


def pick_nut(frets, strings):
    # Returns (nut or None, remaining frets). Needs 3+ lines so we can compare the gaps at both ends.
    if len(frets) < 3:
        return None, frets
    xs = [f['xm'] for f in frets]
    first_gap, last_gap = xs[1] - xs[0], xs[-1] - xs[-2]
    if first_gap >= last_gap:                                  # wider gaps on the left -> headstock on the left
        nut, rest, gap = frets[0], frets[1:], first_gap
        continues = min(st['x1'] for st in strings) <= nut['xm'] - 0.5 * gap
    else:
        nut, rest, gap = frets[-1], frets[:-1], last_gap
        continues = max(st['x2'] for st in strings) >= nut['xm'] + 0.5 * gap
    nut = dict(nut, confidence='medium' if continues else 'low')
    return nut, rest


def _to_original(pts_rot, M, scale):
    inv = cv2.invertAffineTransform(M)
    p = np.asarray(pts_rot, float).reshape(-1, 2) @ inv[:, :2].T + inv[:, 2]
    return (p / scale).round(1).tolist()


def _try_angle(pre, segs, cand):
    # Run strings -> frets -> nut for ONE candidate neck direction and give the result a score for ranking.
    h, w = pre['img'].shape[:2]
    big = max(h, w)
    out = dict(angle=cand['angle'], M=None, strings=[], cv=None, n_frets=0, score=0.0, density=0.0, comb_score=cand['comb_score'])
    fit = find_strings(pre['ridge'], cand)
    if fit is None:
        out.update(status='failed', reason='no 6 evenly spaced, non-crossing strings found along this direction')
        return out
    out.update(M=fit['M'], strings=fit['strings'], cv=fit['cv'], density=fit['density'], xl=fit['xl'], xr=fit['xr'])
    if fit['extent'] < CFG['string_min_extent_frac'] * big or fit['density'] < CFG['string_min_density']:
        out.update(status='failed', reason=f"strings too short ({fit['extent'] / big:.0%} of the image) or too patchy (density {fit['density']:.2f})")
        return out

    strings = fit['strings']
    spacing = float(np.mean(np.diff([c['y0'] for c in strings])))
    sr = to_neck_frame(segs, fit['M'])
    frets, band = find_frets(sr, strings, spacing)
    nut, rest = pick_nut(frets, strings)
    n_lines = len(rest) + (1 if nut else 0)
    # Cross-check: the strings should span about as much width as the fret lines do. If the strings cover much less,
    # we probably grabbed the wrong lines (for example a patch of wood grain).
    all_lines = rest + ([nut] if nut else [])
    ratio = (band['y_bot'] - band['y_top']) / float(np.median([f['span'] for f in all_lines])) if all_lines else None
    # Ranking score: the comb strength, boosted when fret lines cross the strings (a real neck has them, wood grain does not)
    # and by how continuous the string evidence is.
    score = cand['comb_score'] * (1 + 0.3 * min(n_lines, 6)) * fit['density']
    out.update(spacing=spacing, band=band, frets=rest, nut=nut, n_frets=n_lines, band_ratio=ratio, score=score)

    if n_lines >= 2:
        out.update(status='ok', reason=None)
    else:
        out.update(status='partial', reason=f'6 strings found but only {n_lines} fret line(s)')
    return out


def detect_fretboard(bgr, name='image'):
    pre = preprocess(bgr, CFG['max_side'])
    segs = hough_segments(pre['edges'])
    H0, W0 = bgr.shape[:2]
    res = dict(schema_version=1, image=dict(name=name, width=W0, height=H0), status='failed', success=False,
               confidence='none', failure_reason=None, neck_angle_deg=None,
               fretboard=None, strings=[], frets=[], nut=None, diagnostics=dict(n_segments=int(len(segs))))
    cands, _ = neck_candidates(pre['ridge'])
    if not cands:
        res['failure_reason'] = 'no line pattern found (the ridge map is empty)'
        return res

    tries = [_try_angle(pre, segs, c) for c in cands]
    best = max(tries, key=lambda t: t['score'])
    if best['score'] <= 0:                              # nothing valid at all: report the most informative failure
        best = tries[0]

    res['status'], res['success'] = best['status'], best['status'] == 'ok'
    res['failure_reason'], res['neck_angle_deg'] = best['reason'], round(float(best['angle']), 1)
    ratio = best.get('band_ratio')
    res['diagnostics'].update(n_angle_candidates=len(cands), comb_score=round(best['comb_score'], 1),
                              string_density=round(best['density'], 2),
                              string_band_vs_fret_ratio=None if ratio is None else round(float(ratio), 2),
                              string_spacing_cv=None if best['cv'] is None else round(best['cv'], 3),
                              n_fret_lines=best['n_frets'])
    if best['status'] == 'failed':
        return res

    M, scale, strings, sp = best['M'], pre['scale'], best['strings'], best['spacing']
    # Neck extent along the strings comes from where the strings have evidence. Fret lines can extend it a little
    # (frets are sometimes visible where the strings are hidden), but a stray fret far away is ignored.
    xl, xr, basis = best['xl'], best['xr'], 'strings'
    margin = 0.15 * (xr - xl)
    near_x = [f['xm'] for f in best['frets'] + ([best['nut']] if best['nut'] else []) if xl - margin <= f['xm'] <= xr + margin]
    if near_x and (min(near_x) < xl or max(near_x) > xr):
        xl, xr, basis = min(xl, min(near_x)), max(xr, max(near_x)), 'strings+frets'

    top_y = lambda x: string_y(strings[0], x) - sp / 2
    bot_y = lambda x: string_y(strings[-1], x) + sp / 2
    band = best.get('band')

    def fret_ends(f):
        yt, yb = top_y(f['xm']), bot_y(f['xm'])
        return [(f['xm'] + f['k'] * (yt - band['y_mid']), yt), (f['xm'] + f['k'] * (yb - band['y_mid']), yb)]

    res['strings'] = [dict(index=i + 1, p1=_to_original([(xl, string_y(st, xl))], M, scale)[0],
                           p2=_to_original([(xr, string_y(st, xr))], M, scale)[0]) for i, st in enumerate(strings)]
    kept = [f for f in best['frets'] if xl - 1 <= f['xm'] <= xr + 1]
    res['frets'] = [dict(index=i + 1, p1=_to_original([fret_ends(f)[0]], M, scale)[0],
                         p2=_to_original([fret_ends(f)[1]], M, scale)[0]) for i, f in enumerate(kept)]
    if best['nut']:
        e = fret_ends(best['nut'])
        res['nut'] = dict(p1=_to_original([e[0]], M, scale)[0], p2=_to_original([e[1]], M, scale)[0],
                          confidence=best['nut']['confidence'])
    corners = [(xl, top_y(xl)), (xr, top_y(xr)), (xr, bot_y(xr)), (xl, bot_y(xl))]
    res['fretboard'] = dict(polygon=_to_original(corners, M, scale), basis=basis)

    # Confidence: 'ok' only means "geometry was produced". A result is trusted ('high') only if several fret lines
    # were found, the neck spans a large part of the image, the strings cover a sensible share of the neck width,
    # AND the strings show continuous line evidence. Small patches of background texture fail this.
    poly = np.array(res['fretboard']['polygon'])
    length_frac = float(np.hypot(*(poly[1] - poly[0]))) / max(W0, H0)
    res['diagnostics']['neck_length_frac'] = round(length_frac, 2)
    ratio_ok = ratio is not None and ratio >= 0.6              # strings must cover a sensible share of the neck width
    if best['status'] == 'ok' and best['n_frets'] >= 4 and length_frac >= 0.4 and ratio_ok and best['density'] >= 0.6:
        res['confidence'] = 'high'
    else:
        res['confidence'] = 'low'
    return res


COL = dict(region=(0, 165, 255), string=(255, 255, 0), fret=(0, 255, 0), nut=(255, 0, 255))   # BGR colors


def draw_result(bgr, res, alpha=0.22):
    # Draw a result dictionary (works on one loaded from JSON too) on top of the original image.
    out = bgr.copy()
    t = max(1, int(round(max(out.shape[:2]) / 500)))
    P = lambda p: (int(round(p[0])), int(round(p[1])))
    if res.get('fretboard'):
        poly = np.array(res['fretboard']['polygon'], np.int32)
        ov = out.copy(); cv2.fillPoly(ov, [poly], COL['region'])
        out = cv2.addWeighted(ov, alpha, out, 1 - alpha, 0)
        cv2.polylines(out, [poly], True, COL['region'], t + 1, cv2.LINE_AA)
    for s in res.get('strings', []):
        cv2.line(out, P(s['p1']), P(s['p2']), COL['string'], t, cv2.LINE_AA)
    for f in res.get('frets', []):
        cv2.line(out, P(f['p1']), P(f['p2']), COL['fret'], t + 1, cv2.LINE_AA)
    if res.get('nut'):
        cv2.line(out, P(res['nut']['p1']), P(res['nut']['p2']), COL['nut'], t + 2, cv2.LINE_AA)
    return out

In [ ]:
# Phase 3 geometry and mapping: copied from notebook 03 (fingerprint 1042900f2f). Do not edit here.
def _unit(v):
    v = np.asarray(v, float)
    n = np.linalg.norm(v)
    return v / n if n > 0 else v


def _mid(line):
    return (line['p1'] + line['p2']) / 2


def signed_dist(line, P, ref_dir):
    # Perpendicular distance from P to the (infinite) line through p1,p2, positive on the side ref_dir points to.
    d = line['p2'] - line['p1']
    n = _unit([-d[1], d[0]])                      # a normal vector (rotate the direction by 90 degrees)
    if n @ ref_dir < 0:
        n = -n
    return float((P - line['p1']) @ n)


def build_neck_model(p2):
    # Turn a Phase 2 result into the objects the mapping needs. Returns None if there are no usable strings.
    if p2.get('status') == 'failed' or len(p2.get('strings', [])) != 6 or not p2.get('fretboard'):
        return None
    S = [dict(p1=np.array(s['p1'], float), p2=np.array(s['p2'], float)) for s in sorted(p2['strings'], key=lambda s: s['index'])]
    cen = [_mid(s) for s in S]
    c_dir = _unit(cen[-1] - cen[0])                                  # across the neck: string 1 -> string 6
    spacing = float(np.mean(np.diff([(c - cen[0]) @ c_dir for c in cen])))
    if spacing <= 1:
        return None
    a_rough = _unit(np.mean([s['p2'] - s['p1'] for s in S], axis=0))  # along the neck (left end -> right end of Phase 2's view)

    lines = [dict(p1=np.array(f['p1'], float), p2=np.array(f['p2'], float), kind='fret', virtual=False) for f in p2.get('frets', [])]
    nut_conf = None
    if p2.get('nut'):
        lines.append(dict(p1=np.array(p2['nut']['p1'], float), p2=np.array(p2['nut']['p2'], float), kind='nut', virtual=False,
                          conf=p2['nut'].get('confidence')))
    model = dict(strings=S, c_dir=c_dir, spacing=spacing, polygon=np.array(p2['fretboard']['polygon'], float),
                 boundaries=[], b_dir=None, orientation='none', nut_conf=None, n_virtual=0)
    if len(lines) < 2:
        return model                                                  # strings only: we can name the string but not the fret

    lines.sort(key=lambda l: _mid(l) @ a_rough)                       # left to right along the neck
    # Decide which end is the headstock end. Prefer the nut; otherwise the end with wider gaps.
    nut_idx = [i for i, l in enumerate(lines) if l['kind'] == 'nut']
    pos = [_mid(l) @ a_rough for l in lines]
    gaps = np.diff(pos)
    orientation = 'unknown'
    if nut_idx and nut_idx[0] in (0, len(lines) - 1):
        if nut_idx[0] == len(lines) - 1:
            lines = lines[::-1]
        orientation = 'nut'
        model['nut_conf'] = lines[0].get('conf')
    elif len(lines) >= 3:
        if gaps[0] < gaps[-1]:
            lines = lines[::-1]
        orientation = 'gap_trend'
    # Repair missed frets with the 12-tone spacing rule (gaps should shrink by ~0.944 per fret).
    ordered, unit_gap = [lines[0]], None
    r12 = MAPCFG['fret_ratio']
    for j in range(1, len(lines)):
        g = float(np.linalg.norm(_mid(lines[j]) - _mid(lines[j - 1])))
        if j == 1 and orientation != 'unknown' and len(lines) >= 3:
            # The FIRST gap has no earlier gap to compare with, so compare it with the NEXT one. If a gap holds mm+1 real
            # frets, its width relative to the next region is (1 + r + ... + r^mm) / r^(mm+1): 1.06, 2.2, 3.4, ...
            g2 = float(np.linalg.norm(_mid(lines[2]) - _mid(lines[1])))
            R = g / max(g2, 1e-6)
            expect = [sum(r12 ** i for i in range(mm + 1)) / r12 ** (mm + 1) for mm in range(5)]
            mm = int(np.argmin([abs(np.log(max(R, 1e-6) / e)) for e in expect]))
            if mm >= 1 and R > 1.5:
                tot = sum(r12 ** i for i in range(mm + 1))
                for t in np.cumsum([r12 ** i for i in range(mm)]) / tot:     # missed frets sit at the 12-tone proportions
                    ordered.append(dict(p1=lines[0]['p1'] * (1 - t) + lines[1]['p1'] * t,
                                        p2=lines[0]['p2'] * (1 - t) + lines[1]['p2'] * t, kind='fret', virtual=True))
                    model['n_virtual'] += 1
                ordered.append(lines[1])
                unit_gap = g * r12 ** mm / tot
                continue
        m = 1
        if unit_gap is not None and orientation != 'unknown':
            expected = unit_gap * MAPCFG['fret_ratio']
            m = int(np.clip(round(g / expected), 1, 3))
            if m < 2 or g < 1.5 * expected:
                m = 1
        for i in range(1, m):                                         # virtual boundaries, evenly spread inside the gap
            t = i / m
            ordered.append(dict(p1=lines[j - 1]['p1'] * (1 - t) + lines[j]['p1'] * t,
                                p2=lines[j - 1]['p2'] * (1 - t) + lines[j]['p2'] * t, kind='fret', virtual=True))
            model['n_virtual'] += 1
        ordered.append(lines[j])
        unit_gap = g / m
    model.update(boundaries=ordered, orientation=orientation,
                 b_dir=_unit(_mid(ordered[-1]) - _mid(ordered[0])))
    # Irregular spacing check: a real fret region is only ~6% narrower than the one before it. A region much wider than
    # its neighbours hides a missed fret (or two); a much narrower one means a spurious line. We cannot name the fret there.
    widths = [float(np.linalg.norm(_mid(ordered[k]) - _mid(ordered[k - 1]))) for k in range(1, len(ordered))]   # slot k = widths[k-1]
    suspect = {}
    for k in range(1, len(ordered)):
        if ordered[k]['virtual'] or ordered[k - 1]['virtual']:
            continue
        nb = [widths[j - 1] for j in (k - 1, k + 1) if 1 <= j <= len(widths)]
        if not nb:
            continue
        ratio = widths[k - 1] / max(float(np.mean(nb)), 1e-6)
        if ratio > MAPCFG['wide_region_ratio']:
            suspect[k] = f'region {ratio:.1f}x wider than its neighbours: a fret line was probably missed'
        elif ratio < MAPCFG['narrow_region_ratio']:
            suspect[k] = f'region only {ratio:.2f}x as wide as its neighbours: a spurious fret line is likely'
    model['region_suspect'] = suspect
    return model


def locate_string(model, P):
    # Which string is P closest to, and how clearly? Uses the real (possibly fanned-out) string lines.
    S, c_dir = model['strings'], model['c_dir']
    sd = [signed_dist(s, P, c_dir) for s in S]                       # positive = on the string-6 side of that string
    crossing = any(sd[i] <= sd[i + 1] for i in range(5))              # strings must stay in order at this point
    m = sum(d > 0 for d in sd)                                        # number of strings P is past
    if m == 0:                                                        # above string 1
        d1, gap = -sd[0], sd[0] - sd[1]
        near, d2 = 1, d1 + max(gap, 1e-6)
        outside = d1 > MAPCFG['outside_frac'] * max(gap, 1e-6)
        pos = 1 - d1 / max(gap, 1e-6)
    elif m == 6:                                                      # below string 6
        d1, gap = sd[5], sd[4] - sd[5]
        near, d2 = 6, d1 + max(gap, 1e-6)
        outside = d1 > MAPCFG['outside_frac'] * max(gap, 1e-6)
        pos = 6 + d1 / max(gap, 1e-6)
    else:                                                             # between string m and string m+1
        da, db = sd[m - 1], -sd[m]
        near = m if da <= db else m + 1
        d1, d2, gap = min(da, db), max(da, db), da + db
        outside = False
        pos = m + da / max(da + db, 1e-6)
    margin = (d2 - d1) / max(d2 + d1, 1e-6)
    return dict(index=int(near), position=float(pos), margin=float(margin), dist_px=float(d1), outside=bool(outside),
                crossing=bool(crossing), gap_px=float(gap))


def locate_fret(model, P):
    # Which fret region is P in (between which two boundaries), and how far from the nearer boundary?
    B, b_dir = model['boundaries'], model['b_dir']
    sd = [signed_dist(b, P, b_dir) for b in B]                       # positive = on the body side of that line
    k = sum(d > 0 for d in sd)                                        # number of boundaries P is past
    if k == 0:
        return dict(region='before_first_line', slot=None, margin=0.0, virtual_edge=False)
    if k == len(B):
        return dict(region='after_last_line', slot=None, margin=0.0, virtual_edge=False)
    dl, dr = sd[k - 1], -sd[k]                                        # distances to the boundary behind and ahead of P
    width = max(dl + dr, 1e-6)
    return dict(region='inside', slot=int(k), margin=float(min(dl, dr) / width), frac=float(dl / width),
                virtual_edge=bool(B[k - 1]['virtual'] or B[k]['virtual']), suspect=model['region_suspect'].get(int(k)))


def snap_to_string(model, P, index):
    # Foot of the perpendicular from P onto string `index`: the "mapped position" we draw.
    s = model['strings'][index - 1]
    d = _unit(s['p2'] - s['p1'])
    return (s['p1'] + ((P - s['p1']) @ d) * d)

def select_fretting_hand(hands, model):
    # Pick the hand most likely to be the fretting hand. Returns (index or None, number of other hands).
    if not hands['hands']:
        return None, 0
    best, best_key = 0, None
    for i, hd in enumerate(hands['hands']):
        near = 0
        if model is not None:
            poly = model['polygon'].astype(np.float32)
            for name in ('Index', 'Middle', 'Ring', 'Pinky'):
                t = hd['tips'][name]
                # signed distance to the polygon: > 0 inside, < 0 outside (in pixels)
                dist = cv2.pointPolygonTest(poly, (float(t['x_px']), float(t['y_px'])), True)
                near += dist >= -MAPCFG['near_neck_frac'] * model['spacing']
        key = (near, hd['score'])
        if best_key is None or key > best_key:
            best, best_key = i, key
    return best, len(hands['hands']) - 1


def map_fingertip(model, p2, finger, tip, hand_score):
    P = np.array([tip['x_px'], tip['y_px']], float)
    r = dict(finger=finger, landmark_id=tip['id'], x_px=tip['x_px'], y_px=tip['y_px'], proximity_only=True,
             status='no_geometry', decision='rejected', reason='', string_index=None, string_position=None,
             string_margin=None, fret_slot=None, fret_number=None, fret_margin=None, fret_frac=None, neck_coord=None,
             confidence=0.0, snap_xy=None)
    if model is None:
        r['reason'] = f"Phase 2 found no usable strings ({p2.get('failure_reason')})"
        return r
    s = locate_string(model, P)
    r.update(string_index=s['index'], string_position=round(s['position'], 2), string_margin=round(s['margin'], 2))
    if s['crossing']:
        r.update(status='ambiguous_string', reason='detected strings cross at this point; geometry unreliable here')
        return r
    if s['outside']:
        r.update(status='outside_fretboard', reason='beyond the outer string', string_index=None)
        return r
    f = locate_fret(model, P) if model['boundaries'] else None
    if f is not None and f['region'] == 'before_first_line':
        if model['nut_conf'] == 'medium':                              # a believable nut: beyond it is the headstock
            r.update(status='outside_fretboard', reason='behind the nut (headstock side)')
        else:                                                          # nut not confirmed: more fretboard may lie beyond
            r.update(status='beyond_detected_frets', reason='before the first detected line and the nut is not confirmed; region unknown')
        return r
    if f is not None and f['region'] == 'after_last_line':
        r.update(status='beyond_detected_frets', reason='past the last detected fret line; region width unknown')
        return r
    s_ok = s['margin'] >= MAPCFG['string_margin_min']
    if f is None:
        status = 'string_only' if s_ok else 'ambiguous_string'
        f_clear = 1.0
    else:
        f_ok = f['margin'] >= MAPCFG['fret_margin_min'] and not f['suspect']
        status = 'mapped' if (s_ok and f_ok) else 'ambiguous_both' if (not s_ok and not f_ok) else 'ambiguous_string' if not s_ok else 'ambiguous_fret'
        f_clear = float(np.clip(2 * f['margin'], 0, 1)) * (0.5 if f['virtual_edge'] else 1.0)
        r.update(fret_slot=f['slot'], fret_margin=round(f['margin'], 2),
                 fret_number=f['slot'] if model['nut_conf'] == 'medium' else None,
                 fret_frac=round(f['frac'], 3),                       # 0 = on the headstock-side line, 1 = on the body-side line
                 neck_coord=round(f['slot'] - 1 + f['frac'], 3))      # continuous position along the neck, in fret-region units
    gq = MAPCFG['geometry_quality'].get(p2.get('confidence'), 0.3)
    conf = gq * hand_score * float(np.sqrt(np.clip(s['margin'], 0, 1) * f_clear))
    r.update(status=status, confidence=round(float(conf), 3), snap_xy=[round(float(v), 1) for v in snap_to_string(model, P, s['index'])])
    if status.startswith('ambiguous'):
        r['reason'] = 'too close to the boundary between two ' + ('strings' if status == 'ambiguous_string' else 'fret regions' if status == 'ambiguous_fret' else 'strings and fret lines')
        if f is not None and f['suspect'] and status in ('ambiguous_fret', 'ambiguous_both'):
            r['reason'] = f"irregular fret spacing here ({f['suspect']})"
    elif finger == 'Thumb':
        r['reason'] = 'thumb usually rests behind the neck'
    elif p2.get('confidence') != 'high' and not MAPCFG['allow_low_geometry']:
        r.update(decision='tentative', reason='Phase 2 geometry is low confidence')
    elif conf < MAPCFG['accept_conf']:
        r.update(decision='tentative', reason=f"confidence {conf:.2f} below {MAPCFG['accept_conf']}")
    else:
        r.update(decision='accepted', reason='')
    return r


def map_image(image_id, bgr, hands, p2, p2_source='recompute', coord_problems=None):
    # Everything this needs is passed in (no notebook globals), so other notebooks can call it per image.
    model = build_neck_model(p2)
    h_idx, n_other = select_fretting_hand(hands, model)
    bbox = None
    if h_idx is not None:                                             # box around all 21 landmarks of the chosen hand
        lm = np.array(hands['hands'][h_idx]['landmarks_px'])
        bbox = [round(float(v), 1) for v in (lm[:, 0].min(), lm[:, 1].min(), lm[:, 0].max(), lm[:, 1].max())]
    out = dict(schema_version=2, image=dict(id=image_id, width=bgr.shape[1], height=bgr.shape[0]),
               phase1=dict(n_hands=len(hands['hands']), fretting_hand_index=h_idx, n_other_hands=n_other, hand_bbox=bbox),
               phase2=dict(source=p2_source, status=p2['status'], confidence=p2['confidence'], failure_reason=p2.get('failure_reason'),
                           neck_angle_deg=p2.get('neck_angle_deg')),
               geometry=dict(usable=model is not None, n_fret_boundaries=0 if not model else len(model['boundaries']),
                             n_virtual_boundaries=0 if not model else model['n_virtual'],
                             orientation='none' if not model else model['orientation'],
                             nut_confidence=None if not model else model['nut_conf'],
                             string_index_convention='1..6 in Phase 2 order (one side of the neck to the other); NOT guitar tuning',
                             fret_slot_convention='1 = region after the first detected headstock-side line; fret_number only when a medium-confidence nut anchors it'),
               coordinate_problems=coord_problems or [], fingertips=[], category='')
    if h_idx is None:
        out['category'] = 'no_hand'
        return out, model
    hd = hands['hands'][h_idx]
    for name in FINGERTIPS.values():
        r = map_fingertip(model, p2, name, hd['tips'][name], hd['score'])
        r['hand_score'] = hd['score']
        out['fingertips'].append(r)
    dec = [r['decision'] for r in out['fingertips']]
    stat = [r['status'] for r in out['fingertips']]
    if model is None:
        out['category'] = 'geometry_failed'
    elif 'accepted' in dec:
        out['category'] = 'success'
    elif 'tentative' in dec:
        out['category'] = 'tentative_low_geometry'
    elif any(s.startswith('ambiguous') for s in stat):
        out['category'] = 'ambiguous'
    else:
        out['category'] = 'outside_or_rejected'
    return out, model

In [ ]:
FINGERS = ['Index', 'Middle', 'Ring', 'Pinky', 'Thumb']
FRET4 = ['Index', 'Middle', 'Ring', 'Pinky']
DEC = {'accepted': 2, 'tentative': 1, 'rejected': 0}
IN_NECK = {'mapped', 'string_only', 'ambiguous_string', 'ambiguous_fret', 'ambiguous_both'}
nan = float('nan')


def crop_box(res):
    # Box (in original pixels) around the fretboard and the hand, padded and clipped to the image.
    W, H = res['image']['width'], res['image']['height']
    pts = []
    poly = (res.get('phase2_geometry') or {}).get('fretboard')
    kinds = []
    if poly:
        pts += [tuple(p) for p in poly['polygon']]; kinds.append('neck')
    bb = res['phase1'].get('hand_bbox')
    if bb:
        pts += [(bb[0], bb[1]), (bb[2], bb[3])]; kinds.append('hand')
    if not pts:
        return 0, 0, W, H, 'full'
    a = np.array(pts, float)
    x0, y0, x1, y1 = a[:, 0].min(), a[:, 1].min(), a[:, 0].max(), a[:, 1].max()
    px, py = 0.12 * (x1 - x0), 0.12 * (y1 - y0)
    box = [int(max(0, x0 - px)), int(max(0, y0 - py)), int(min(W, x1 + px)), int(min(H, y1 + py))]
    if box[2] - box[0] < 32 or box[3] - box[1] < 32:
        return 0, 0, W, H, 'full'
    return (*box, '+'.join(kinds))


def feature_row(res):
    p2, g = res['phase2'], res['geometry']
    tips = {t['finger']: t for t in res['fingertips']}
    row = dict(id=res['image']['id'], chord=res['chord'], split=res['split'], row=res['row'],
               width=res['image']['width'], height=res['image']['height'],
               hand_found=int(res['phase1']['fretting_hand_index'] is not None), n_hands=res['phase1']['n_hands'],
               geom_usable=int(g['usable']), geom_status=p2['status'], geom_conf=p2['confidence'],
               geom_conf_num={'high': 1.0, 'low': 0.6}.get(p2['confidence'], 0.0), category=res['category'],
               n_accepted=sum(t['decision'] == 'accepted' for t in res['fingertips']),
               nut_conf=g.get('nut_confidence') or 'none', n_boundaries=g['n_fret_boundaries'], n_virtual=g['n_virtual_boundaries'],
               hand_score=tips['Index']['hand_score'] if tips else nan)
    for f in FINGERS:
        t = tips.get(f)
        on_neck = t is not None and t['status'] in IN_NECK
        row[f'{f}_in'] = int(on_neck)
        row[f'{f}_sp'] = t['string_position'] if (t and (on_neck or t['status'] == 'beyond_detected_frets')) else nan
        row[f'{f}_nc'] = t['neck_coord'] if (on_neck and t['neck_coord'] is not None) else nan
        row[f'{f}_conf'] = t['confidence'] if t else nan
        row[f'{f}_dec'] = DEC[t['decision']] if t else nan
    for a_i, a in enumerate(FRET4):                      # differences between the fretting fingers: chords are shapes
        for b in FRET4[a_i + 1:]:
            row[f'd_sp_{a}_{b}'] = row[f'{a}_sp'] - row[f'{b}_sp']
            row[f'd_nc_{a}_{b}'] = row[f'{a}_nc'] - row[f'{b}_nc']
    sp = [row[f'{f}_sp'] for f in FRET4]; nc = [row[f'{f}_nc'] for f in FRET4]
    sp_min = np.nanmin(sp) if not np.all(np.isnan(sp)) else nan
    nc_min = np.nanmin(nc) if not np.all(np.isnan(nc)) else nan
    for f in FRET4:
        row[f'{f}_sp_rel'] = row[f'{f}_sp'] - sp_min
        row[f'{f}_nc_rel'] = row[f'{f}_nc'] - nc_min
    row['n_in_neck'] = sum(row[f'{f}_in'] for f in FRET4)
    row['features_usable'] = int(row['hand_found'] and row['n_in_neck'] >= 2)
    x0, y0, x1, y1, kind = crop_box(res)
    row.update(crop_x0=x0, crop_y0=y0, crop_x1=x1, crop_y1=y1, crop_kind=kind)
    return row

In [ ]:
def root_of(name): return name[:-1] if name.endswith('m') else name
def is_minor(name): return name.endswith('m')


def metrics(y, prob, names):
    # y: true class indices, prob: [N, C] probabilities. Returns a dict of the metrics described above.
    y = np.asarray(y); pred = prob.argmax(1)
    top3 = np.mean([yy in np.argsort(-p)[:3] for yy, p in zip(y, prob)]) if len(y) else np.nan
    roots = np.array([root_of(n) for n in names]); minor = np.array([is_minor(n) for n in names])
    cls = np.unique(y)
    f1 = []
    for c in cls:
        tp, fp, fn = ((pred == c) & (y == c)).sum(), ((pred == c) & (y != c)).sum(), ((pred != c) & (y == c)).sum()
        f1.append(2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0)
    return dict(n=int(len(y)), acc=float((pred == y).mean()) if len(y) else np.nan, top3=float(top3),
                bal_acc=float(np.mean([(pred[y == c] == c).mean() for c in cls])) if len(cls) else np.nan,
                macro_f1=float(np.mean(f1)) if f1 else np.nan,
                root_acc=float((roots[pred] == roots[y]).mean()) if len(y) else np.nan,
                mm_acc=float((minor[pred] == minor[y]).mean()) if len(y) else np.nan)


def _components(n, ii, jj):
    parent = list(range(n))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    for a, b in zip(ii, jj):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
    return np.array([find(i) for i in range(n)])


def _leader_clusters(sub, t):
    # Chain-free grouping: repeatedly take the image with the most unassigned near-neighbours and group it WITH those
    # neighbours only (neighbours of neighbours are not pulled in), so a slow drift of look-alikes cannot become one blob.
    m = len(sub); free = np.ones(m, bool); lab = -np.ones(m, int); adj = sub >= t; k = 0
    while free.any():
        deg = (adj & free[None, :]).sum(1) * free
        seed = int(deg.argmax()) if deg.max() > 0 else int(np.where(free)[0][0])
        grp = free & (adj[seed] | (np.arange(m) == seed))
        lab[grp] = k; free &= ~grp; k += 1
    return lab


def group_by_similarity(sim, tau, max_frac=0.12, step=0.02, max_tau=0.995):
    # Connected components of the graph "similarity >= tau". An oversized group is re-split with a stricter tau; if tau
    # cannot be tightened any further (a continuous chain of look-alikes), it is broken with chain-free leader clustering.
    # NOTE: breaking a chain can put neighbouring frames in different groups, so a little leakage can remain.
    n = len(sim); groups = -np.ones(n, int); nxt = 0
    stack = [(np.arange(n), tau)]
    while stack:
        idx, t = stack.pop()
        sub = sim[np.ix_(idx, idx)]
        ii, jj = np.where(np.triu(sub >= t, 1))
        lab = _components(len(idx), ii, jj)
        for c in np.unique(lab):
            members = idx[lab == c]
            if len(members) <= max_frac * n:
                groups[members] = nxt; nxt += 1
            elif t + step <= max_tau:
                stack.append((members, t + step))                 # still too big: tighten the threshold and look again
            else:
                sub2 = sim[np.ix_(members, members)]
                lab2 = _leader_clusters(sub2, t)
                for c2 in np.unique(lab2):
                    groups[members[lab2 == c2]] = nxt; nxt += 1
    return groups


def fuse(p_cnn, p_feat, feat_conf, alpha, w_max=0.9):
    # Blend the two probability tables. Where features are missing (feat_conf is NaN) the CNN is used alone.
    fc = np.nan_to_num(np.asarray(feat_conf, float), nan=0.0)
    w = np.clip(alpha * fc, 0.0, w_max)[:, None]
    pf = np.where(np.isnan(p_feat), 0.0, p_feat)
    # the blend applies only where there is BOTH a feature confidence and a Model B probability row; elsewhere the CNN stands alone
    has = (~np.isnan(np.asarray(feat_conf, float)) & ~np.isnan(p_feat).all(axis=1))[:, None]
    return np.where(has, (1 - w) * p_cnn + w * pf, p_cnn)


def coverage_curve(prob, y, thresholds):
    # For each confidence threshold: share of images answered, and accuracy on the answered ones.
    conf, pred = prob.max(1), prob.argmax(1)
    out = []
    for t in thresholds:
        keep = conf >= t
        out.append((t, keep.mean(), (pred[keep] == y[keep]).mean() if keep.any() else np.nan))
    return pd.DataFrame(out, columns=['threshold', 'coverage', 'accuracy'])
print('helpers ready')

## 2. Model loading and one-image prediction
**What:** `predict_image(bgr)` is the **whole system** for one photo: Phase 1 (hand), Phase 2 (neck), Phase 3 (finger positions), feature row, crop, the CNN probabilities, Model B probabilities, the blend, and the final answer.
- `chord` is the most likely chord, or `None` (shown as "not sure") when the blended top probability is below the abstention threshold chosen in 04b.
- `used` says which parts contributed: `cnn only` (no usable fingertip features) or `cnn + features`.
**Technique: softmax probabilities.** Each model ends with 14 scores that are turned into probabilities summing to 1. The highest one is the guess and its value is the confidence. A model can be confidently wrong, which is why we also measure coverage vs accuracy instead of trusting the confidence blindly.

In [ ]:
import torch.nn as nn
import joblib
from torchvision import models, transforms as T
from PIL import Image

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
S = SET['img_size']
eval_tf = T.Compose([T.Resize((S, S)), T.ToTensor(), T.Normalize(SET['mean'], SET['std'])])


def make_landmarker():
    # Exactly the Phase 1 settings: image mode, up to 2 hands, all confidences 0.3 (`detect_hands` uses the global LANDMARKER).
    return vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=MODEL_PATH), running_mode=vision.RunningMode.IMAGE, num_hands=2,
        min_hand_detection_confidence=0.3, min_hand_presence_confidence=0.3, min_tracking_confidence=0.3))
LANDMARKER = make_landmarker()


def load_cnn(path):
    ck = torch.load(path, map_location=dev)
    m = models.efficientnet_b0(weights=None)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, len(NAMES))
    m.load_state_dict(ck['model'])
    return m.to(dev).eval(), ck['mode']


CNN = {}
for key, fname in (('A1', 'A1_group.pt'), ('A2', 'A2_crop.pt')):
    if (MODELS / fname).exists():
        CNN[key] = load_cnn(MODELS / fname)
print('loaded CNNs:', {k: v[1] for k, v in CNN.items()})
MODEL_B = joblib.load(MODELS / 'model_b.joblib') if (MODELS / 'model_b.joblib').exists() else None
print('Model B (feature trees):', 'loaded' if MODEL_B else 'not available (it was not trained in 04b)')


@torch.no_grad()
def cnn_probs(key, pil):
    model, mode = CNN[key]
    x = eval_tf(pil).unsqueeze(0).to(dev)
    return torch.softmax(model(x).float(), 1)[0].cpu().numpy()


def proba_full(model, X):
    p = model.predict_proba(X); out = np.zeros((len(X), len(NAMES))); out[:, model.classes_] = p
    return out


def feat_conf_row(row):
    # confidence in the fingertip features: Phase 2 geometry confidence x mean Phase 3 confidence of the fingertips on the neck
    if not row['features_usable']:
        return np.nan
    c = [row[f'{f}_conf'] for f in ('Index', 'Middle', 'Ring', 'Pinky') if row[f'{f}_in'] == 1 and not np.isnan(row[f'{f}_conf'])]
    return row['geom_conf_num'] * float(np.mean(c)) if c else np.nan


def run_phases(bgr, image_id='photo', p2=None):
    # Phases 1-3 for one photo (pass p2 to reuse an already detected fretboard)
    hands = detect_hands(bgr, image_id)
    p2 = p2 if p2 is not None else detect_fretboard(bgr, image_id)
    res, _ = map_image(image_id, bgr, hands, p2, 'recompute', check_coordinates(image_id, bgr, hands, p2))
    res.update(chord='?', split='own', row=0, phase2_geometry=p2)
    return res, feature_row(res), p2


def predict_from(bgr, row):
    # models only (Phases 1-3 already done): CNN on the right input, Model B on the feature row, blend, abstain
    pil_full = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    pil_crop = pil_full.crop((int(row['crop_x0']), int(row['crop_y0']), int(row['crop_x1']), int(row['crop_y1']))) if row['crop_kind'] != 'full' else pil_full
    probs = {k: cnn_probs(k, pil_crop if CNN[k][1] == 'crop' else pil_full) for k in CNN}
    p_cnn = probs[BASE if BASE in probs else next(iter(probs))][None, :]
    p_feat = np.full((1, len(NAMES)), np.nan)
    fc = feat_conf_row(row)
    if MODEL_B is not None and not np.isnan(fc):
        p_feat = proba_full(MODEL_B['trees'], np.array([[row[c] for c in B_COLS]], float))
    fused = fuse(p_cnn, p_feat, np.array([fc]), ALPHA)[0]
    top = int(fused.argmax())
    used = 'cnn + features' if (not np.isnan(fc) and not np.isnan(p_feat).all() and ALPHA > 0) else 'cnn only'
    return dict(chord=NAMES[top] if fused[top] >= THRESH else None, best=NAMES[top], conf=float(fused[top]), probs=fused,
                p_cnn=p_cnn[0], p_feat=None if np.isnan(p_feat).all() else p_feat[0], used=used, feat_conf=fc)


def predict_image(bgr, image_id='photo'):
    res, row, p2 = run_phases(bgr, image_id)
    out = predict_from(bgr, row)
    out.update(res=res, row=row, p2=p2)
    return out
print('prediction code ready')

## 3. Report on the group-split test set
**What:** the saved probabilities from 04b for the **test images** (a held-out set whose near-duplicate groups never appeared in training), turned into the metrics below. *Top-1* is plain accuracy; *top-3* is "right chord among the 3 best guesses"; *root* counts `Am` for `A` as right; *major/minor* only checks that. *Balanced accuracy* and *macro-F1* weigh every chord equally.
**Subset "with features":** images where Phase 3 produced usable fingertip numbers. This is where Model B and the blend can help at all; everywhere else the system is the CNN alone.
**Coverage vs accuracy:** answering only when confident. The row at the chosen abstention threshold is what the system would do in use.

In [ ]:
P = np.load(MODELS / 'predictions.npz', allow_pickle=True)
yt = P['y_test']; test_ids = list(P['test_ids'])
fused, a1, a2, pb = P['fused_test'], P['a1_test'], P['a2_test'], P['b_test']
has_feat = ~np.isnan(pb).all(axis=1)
print(f'test images: {len(yt)} | with usable fingertip features: {has_feat.sum()} ({has_feat.mean():.0%})')

rows = []
for name, prob in (('A1 whole-image CNN', a1), ('A2 crop CNN', a2), ('FUSION (what the system answers)', fused)):
    for subset, sel in (('all test images', np.ones(len(yt), bool)), ('test images WITH features', has_feat), ('test images WITHOUT features', ~has_feat)):
        if sel.any():
            rows.append(dict(model=name, subset=subset, **{k: round(v, 3) for k, v in metrics(yt[sel], prob[sel], NAMES).items()}))
REPORT = pd.DataFrame(rows); print(REPORT.to_string(index=False))

pred = fused.argmax(1); conf = fused.max(1)
cls_rows = []
for c, n in enumerate(NAMES):
    t = yt == c
    if t.any():
        tp = ((pred == c) & t).sum(); fp = ((pred == c) & ~t).sum()
        cls_rows.append(dict(chord=n, images=int(t.sum()), recall=round(tp / t.sum(), 3), precision=round(tp / (tp + fp), 3) if (tp + fp) else np.nan))
print('\nper chord (fusion):'); print(pd.DataFrame(cls_rows).to_string(index=False))
conf_pairs = collections.Counter((NAMES[a], NAMES[b]) for a, b in zip(yt, pred) if a != b)
print('\nmost common mistakes (true -> predicted):', [(f'{a}->{b}', n) for (a, b), n in conf_pairs.most_common(8)])

cm = np.zeros((len(NAMES), len(NAMES)), int)
for a, b in zip(yt, pred): cm[a, b] += 1
fig, ax = plt.subplots(1, 2, figsize=(13, 5.2))
ax[0].imshow(cm, cmap='Blues'); ax[0].set_xticks(range(len(NAMES)), NAMES, rotation=90); ax[0].set_yticks(range(len(NAMES)), NAMES)
ax[0].set_xlabel('predicted'); ax[0].set_ylabel('true'); ax[0].set_title('confusion matrix (test, fusion)')
for i in range(len(NAMES)):
    for j in range(len(NAMES)):
        if cm[i, j]: ax[0].text(j, i, cm[i, j], ha='center', va='center', fontsize=7, color='white' if cm[i, j] > cm.max() / 2 else 'black')
cv = coverage_curve(fused, yt, np.round(np.linspace(0, 0.95, 20), 2))
ax[1].plot(cv['coverage'], cv['accuracy'], 'o-'); ax[1].set_xlabel('coverage (share of images answered)'); ax[1].set_ylabel('accuracy on answered'); ax[1].set_ylim(0, 1.02); ax[1].grid(alpha=.3)
ax[1].set_title('answer fewer, be right more often')
plt.tight_layout(); plt.show()
k = conf >= THRESH
print(f'\nAt the chosen abstention threshold {THRESH}: the system answers {k.mean():.1%} of test images, with accuracy {(pred[k] == yt[k]).mean():.3f} on those it answers.')
if SET['rehearsal']:
    print('REHEARSAL models: do not read anything into these numbers.')

## 4. Error gallery: why was it wrong?
**What:** test images the system got **wrong**, re-processed through Phases 1-3 and drawn on the original photo. Orange = fretboard region, cyan = strings, green = frets, magenta = nut; fingertip dots are green (accepted by Phase 3), yellow (tentative) or grey (rejected). The title shows `true -> predicted (confidence)` and whether the blend used fingertip features.
**How to read a mistake:**
- *No hand dots at all:* MediaPipe missed the hand, so the CNN had to guess alone.
- *Orange region on the wrong thing:* Phase 2 found the wrong neck.
- *Dots look right and the prediction is a neighbour chord (Am for A):* a genuinely hard major/minor call.
It re-runs Phases 1-3 for each shown image (about 2 seconds each), and loads the pictures from the same Hugging Face dataset.

In [ ]:
from datasets import load_dataset
R = 'hf://datasets/j-asra-j/guitar-chords@refs%2Fconvert%2Fparquet/default/'
_ds = load_dataset('parquet', data_files={'train': [R + 'train/0000.parquet', R + 'train/0001.parquet'],
                                           'validation': R + 'validation/0000.parquet', 'test': R + 'test/0000.parquet'})
HF = {k: _ds[k] for k in ('train', 'validation', 'test')}


def bgr_by_id(iid):
    chord, split, row = iid.rsplit('_', 2)[0], iid.rsplit('_', 2)[1], int(iid.rsplit('_', 1)[1])
    return cv2.cvtColor(np.array(HF[split][row]['image'].convert('RGB')), cv2.COLOR_RGB2BGR)


def overlay(bgr, out):
    img = draw_result(bgr, out['p2'], alpha=0.18)
    s = max(img.shape[:2]) / 640.0
    for t in out['res']['fingertips']:
        c = {'accepted': (0, 200, 0), 'tentative': (0, 255, 255)}.get(t['decision'], (160, 160, 160))
        p = (int(round(t['x_px'])), int(round(t['y_px'])))
        cv2.circle(img, p, max(3, int(6 * s)), c, -1, cv2.LINE_AA); cv2.circle(img, p, max(3, int(6 * s)) + 1, (0, 0, 0), 1, cv2.LINE_AA)
    r = out['row']
    return img[int(r['crop_y0']):int(r['crop_y1']), int(r['crop_x0']):int(r['crop_x1'])]


wrong = [i for i in range(len(yt)) if pred[i] != yt[i]]
rng = random.Random(0); rng.shuffle(wrong)
print(f'{len(wrong)} wrong test images of {len(yt)}; showing up to 9')
ims, ts = [], []
for i in wrong[:9]:
    iid = test_ids[i]; bgr = bgr_by_id(iid)
    out = predict_image(bgr, iid)
    ims.append(overlay(bgr, out))
    ts.append(f"{iid}\ntrue {NAMES[yt[i]]} -> {out['best']} ({out['conf']:.2f}) | {out['used']}" + ('' if out['chord'] else ' | NOT SURE'))
if ims:
    show_images(ims, ts, cols=3, size=4.6)

## 5. Your own photos (the number that matters)
**What:** a zip of your photos, one folder per chord using the exact chord names (`Am/`, `C/`, `Gm/` ...). Each photo goes through the full system. The models **never saw these photos**, so this score tells you whether the system works on *your* guitar, lighting and camera angle.
**How to take them (this matters):** the same camera and angle you will use in the demo; the chord fully formed; both the fretting hand and the neck in view; vary the lighting a little; 10-20 photos per chord is a good start; take some right after re-fretting so they are not near-identical.
**Set up:** zip the chord folders into `own_test.zip` and either put it at `PROJECT_DIR/own_test.zip` or set `ASK_FOR_OWN_PHOTOS = True` to get a file picker. If there is no zip, the cell prints this and skips.
**Reading the result:** a big gap between section 3 and this section is the honest measure of how much the dataset's look differs from your setup. That gap is what the next phase (fine-tuning with some of your photos) is for.

In [ ]:
ASK_FOR_OWN_PHOTOS = False
OWN_ZIP = Path(PROJECT_DIR) / 'own_test.zip'
if not OWN_ZIP.exists() and ASK_FOR_OWN_PHOTOS:
    from google.colab import files
    up = files.upload(); OWN_ZIP.write_bytes(up[[n for n in up if n.lower().endswith('.zip')][0]])

OWN = []
if OWN_ZIP.exists():
    d = Path(PROJECT_DIR) / 'own_test'; shutil.rmtree(d, ignore_errors=True)
    with zipfile.ZipFile(OWN_ZIP) as z: z.extractall(d)
    for f in sorted(d.rglob('*')):
        if f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp'} and '__MACOSX' not in f.parts and f.parent.name in NAMES:
            bgr = cv2.imread(str(f))
            if bgr is not None:
                OWN.append(dict(path=f, chord=f.parent.name, bgr=bgr))
    print(f'{len(OWN)} own photos found; chords: {pd.Series([o["chord"] for o in OWN]).value_counts().sort_index().to_dict()}')
    unknown = {f.parent.name for f in d.rglob('*') if f.is_file() and f.parent.name not in NAMES and f.parent != d}
    if unknown: print('ignored folders (not a known chord name):', sorted(unknown))
else:
    print('No own_test.zip yet, so this section is skipped. See the text above for how to take the photos.')

if OWN:
    t0 = time.time(); rows, outs = [], []
    for o in OWN:
        out = predict_image(o['bgr'], o['path'].stem); outs.append(out)
        rows.append(dict(photo=o['path'].name, true=o['chord'], pred=out['best'], conf=round(out['conf'], 3), answered=out['chord'] is not None,
                         used=out['used'], hand_found=int(out['row']['hand_found']), geometry=out['row']['geom_conf'],
                         cnn_only_pred=NAMES[int(out['p_cnn'].argmax())]))
    OWN_DF = pd.DataFrame(rows)
    y = np.array([NAMES.index(c) for c in OWN_DF['true']]); prob = np.vstack([o['probs'] for o in outs]); pc = np.vstack([o['p_cnn'] for o in outs])
    print(f'processed {len(OWN)} photos in {time.time() - t0:.0f}s')
    print('FUSION on your photos :', {k: round(v, 3) for k, v in metrics(y, prob, NAMES).items()})
    print('CNN alone on your photos:', {k: round(v, 3) for k, v in metrics(y, pc, NAMES).items()})
    ans = OWN_DF['answered'].values
    print(f"answered {ans.mean():.0%} of photos; accuracy on answered {(OWN_DF['pred'][ans] == OWN_DF['true'][ans]).mean():.3f}" if ans.any() else 'abstained on every photo')
    print('hand found in', f"{OWN_DF['hand_found'].mean():.0%}", 'of your photos (fingertip features exist only for those)')
    print(OWN_DF.groupby('true').apply(lambda g: pd.Series({'photos': len(g), 'correct': int((g['pred'] == g['true']).sum())}), include_groups=False).T.to_string())
    bad = [i for i, o in enumerate(OWN) if OWN_DF.loc[i, 'pred'] != OWN_DF.loc[i, 'true']][:9]
    if bad:
        show_images([overlay(OWN[i]['bgr'], outs[i]) for i in bad],
                    [f"{OWN[i]['path'].name}\ntrue {OWN_DF.loc[i, 'true']} -> {OWN_DF.loc[i, 'pred']} ({OWN_DF.loc[i, 'conf']})" for i in bad], cols=3, size=4.6)

## 6. Live demo with your webcam
**What:** the whole system on live video.
1. **Lock the fretboard.** Phase 2 takes about 1.7 s per frame, far too slow to run on every frame, and the guitar hardly moves. So the first frame (or when you press **Re-lock fretboard**) detects the neck once and every later frame **reuses** it. If you move the guitar, press Re-lock.
2. **Every frame:** MediaPipe in **video mode** (which tracks the hand between frames), then Phase 3 with the locked neck, the feature row, the crop, the CNN, Model B, and the blend.
3. **Smoothing:** single frames are noisy. The last `SMOOTH_FRAMES` probability vectors are averaged, then the abstention threshold decides whether to show a chord or **not sure**.
**Honest expectation:** the video goes browser to Colab and back every frame, so expect a few frames per second, not smooth video. Hold the chord steady for a second or two before reading the answer.
**Test without a webcam:** `simulate_stream(images)` pushes still images through the exact same code, so the pipeline can be checked without a camera.

In [ ]:
from mediapipe.tasks import python as _mp_python
from mediapipe.tasks.python import vision as _vision

SMOOTH_FRAMES = 8


class VideoLandmarker:
    # Lets the unchanged `detect_hands` (which calls LANDMARKER.detect) run in MediaPipe's VIDEO mode, which tracks hands over time.
    def __init__(self):
        self.lm = _vision.HandLandmarker.create_from_options(_vision.HandLandmarkerOptions(
            base_options=_mp_python.BaseOptions(model_asset_path=MODEL_PATH), running_mode=_vision.RunningMode.VIDEO, num_hands=2,
            min_hand_detection_confidence=0.3, min_hand_presence_confidence=0.3, min_tracking_confidence=0.3))
        self.t0, self.last = time.time(), -1
    def detect(self, mp_img):
        ts = max(int((time.time() - self.t0) * 1000), self.last + 1)        # timestamps must strictly increase
        self.last = ts
        return self.lm.detect_for_video(mp_img, ts)


LIVE = dict(p2=None, hist=collections.deque(maxlen=SMOOTH_FRAMES), n=0, t_last=time.time(), fps=0.0)
IMAGE_LANDMARKER = LANDMARKER            # still photos always use the image-mode landmarker
VIDEO_LANDMARKER = VideoLandmarker()     # the live demo uses the tracking (video-mode) one


def live_reset():
    LIVE.update(p2=None, n=0); LIVE['hist'].clear()


def live_step(bgr, relock=False):
    # One frame through the whole system. Returns the annotated frame (BGR), the label and a short info text.
    global LANDMARKER
    LANDMARKER = VIDEO_LANDMARKER                                   # swapped in only for the live frame ...
    try:
        LIVE['n'] += 1
        if relock or LIVE['p2'] is None or (LIVE['p2']['status'] == 'failed' and LIVE['n'] % 20 == 0):
            LIVE['p2'] = detect_fretboard(bgr, 'live'); LIVE['hist'].clear()      # slow (about 2 s) but done only when needed
        res, row, _ = run_phases(bgr, 'live', p2=LIVE['p2'])
    finally:
        LANDMARKER = IMAGE_LANDMARKER                               # ... and restored, so still photos never inherit tracking state
    out = predict_from(bgr, row)
    LIVE['hist'].append(out['probs'])
    avg = np.mean(LIVE['hist'], axis=0); top = int(avg.argmax()); conf = float(avg[top])
    now = time.time(); LIVE['fps'] = 0.8 * LIVE['fps'] + 0.2 / max(now - LIVE['t_last'], 1e-3) if LIVE['n'] > 1 else 0.0; LIVE['t_last'] = now
    img = draw_result(bgr, LIVE['p2'], alpha=0.15)
    s = max(img.shape[:2]) / 640.0
    for t in res['fingertips']:
        c = {'accepted': (0, 200, 0), 'tentative': (0, 255, 255)}.get(t['decision'], (160, 160, 160))
        cv2.circle(img, (int(t['x_px']), int(t['y_px'])), max(3, int(6 * s)), c, -1, cv2.LINE_AA)
    label = f"{NAMES[top]}  {conf:.2f}" if conf >= THRESH else 'not sure'
    cv2.rectangle(img, (0, 0), (img.shape[1], int(46 * s)), (0, 0, 0), -1)
    cv2.putText(img, label, (int(8 * s), int(34 * s)), cv2.FONT_HERSHEY_SIMPLEX, 1.1 * s, (0, 255, 255) if conf >= THRESH else (200, 200, 200), max(2, int(2 * s)), cv2.LINE_AA)
    info = f"{out['used']} | hand {'yes' if row['hand_found'] else 'no'} | neck {row['geom_conf']} | {LIVE['fps']:.1f} fps"
    return img, label, info


def simulate_stream(frames, relock_first=True, show=True):
    # Push still images (BGR arrays) through the live code, as if they were consecutive video frames.
    live_reset(); outs = []
    for k, f in enumerate(frames):
        img, label, info = live_step(f, relock=(relock_first and k == 0)); outs.append((img, label, info))
    if show:
        show_images([o[0] for o in outs[:6]], [f'frame {k + 1}: {o[1]}\n{o[2]}' for k, o in enumerate(outs[:6])], cols=3, size=4.4)
    return outs
print('live code ready (webcam cell below; or call simulate_stream([...images...]))')

In [ ]:
# Live webcam. Click Allow when the browser asks for the camera. Hold the chord, press 'Re-lock fretboard' if the guitar moves.
from google.colab import output
from IPython.display import display, Javascript, JSON


def process_frame(data_url, relock):
    jpg = base64.b64decode(data_url.split(',')[1])
    bgr = cv2.imdecode(np.frombuffer(jpg, np.uint8), cv2.IMREAD_COLOR)
    img, label, info = live_step(bgr, bool(relock))
    ok, buf = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, 80])
    return JSON({'img': 'data:image/jpeg;base64,' + base64.b64encode(buf).decode(), 'label': label, 'info': info})

output.register_callback('notebook.chord_frame', process_frame)
live_reset()

JS = '''
async function run() {
  const box = document.createElement("div");
  const stopBtn = document.createElement("button"); stopBtn.textContent = "Stop camera"; stopBtn.style.cssText = "font-size:16px;padding:6px 14px;margin-right:8px";
  const lockBtn = document.createElement("button"); lockBtn.textContent = "Re-lock fretboard"; lockBtn.style.cssText = "font-size:16px;padding:6px 14px";
  const status = document.createElement("div"); status.style.cssText = "font:16px sans-serif;margin:8px 0";
  const img = document.createElement("img"); img.style.maxWidth = "100%";
  box.append(stopBtn, lockBtn, status, img);
  document.querySelector("#output-area").appendChild(box);
  let stop = false, relock = true;
  stopBtn.onclick = () => { stop = true; }; lockBtn.onclick = () => { relock = true; };
  try {
    const stream = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480}});
    const video = document.createElement("video"); video.srcObject = stream; video.muted = true; await video.play();
    const canvas = document.createElement("canvas"); canvas.width = video.videoWidth; canvas.height = video.videoHeight;
    const ctx = canvas.getContext("2d");
    status.textContent = "Locking the fretboard on the first frame (takes a couple of seconds)...";
    while (!stop) {
      ctx.drawImage(video, 0, 0);
      const r = relock; relock = false;
      const res = await google.colab.kernel.invokeFunction("notebook.chord_frame", [canvas.toDataURL("image/jpeg", 0.7), r], {});
      const d = res.data["application/json"];
      img.src = d.img; status.textContent = d.label + "   (" + d.info + ")";
    }
    stream.getTracks().forEach(t => t.stop()); status.textContent = "Camera stopped.";
  } catch (e) { status.textContent = "Camera error: " + e; }
}
run();
'''
display(Javascript(JS))

In [ ]:
# Check the live pipeline WITHOUT a webcam: feed a few test photos of one chord as if they were video frames
sample_ids = [test_ids[i] for i in range(len(test_ids)) if yt[i] == yt[0]][:4]
frames = [bgr_by_id(i) for i in sample_ids]
print('true chord:', NAMES[yt[0]], '| frames:', sample_ids)
_ = simulate_stream(frames)

## 7. Summary, honest limits, and what comes next
**Save:** the evaluation numbers go into `p4_eval.json` (downloaded as a zip) so they can be compared after the next phase.

**What these results do and do not say**
- The **group-split test** is the cleanest number available from the dataset, but the dataset itself is augmented and visually narrow. Treat it as an upper bound.
- **Your own photos** are the real measure. A big drop there is expected before any fine-tuning.
- Fingertip features exist only for the images where MediaPipe found a hand *and* Phase 3 placed fingers on the neck. Everywhere else the system is the CNN alone, and the report says how large each part is.
- A confidence value is not a guarantee. Use the coverage-vs-accuracy curve, not a single number, to decide when to trust an answer.

**Next phase (fine-tuning), in the order I would try**
1. Add a portion of **your own photos** to training (keeping a separate set for testing).
2. Look at which chords get confused (usually major/minor pairs) and collect more of those.
3. Improve the weakest link the error gallery points to: missed hands, wrong necks, or near-identical chord shapes.
4. Only then tune the CNN (longer training, stronger augmentation, a larger backbone).

In [ ]:
summary = dict(rehearsal=SET['rehearsal'], base_cnn=BASE, alpha=ALPHA, abstain_threshold=THRESH,
               test_report=json.loads(REPORT.to_json(orient='records')),
               own_photos=None if not OWN else dict(n=len(OWN),
                    fusion={k: round(v, 4) for k, v in metrics(y, prob, NAMES).items()}, cnn_only={k: round(v, 4) for k, v in metrics(y, pc, NAMES).items()}))
EVAL_DIR = P4 / 'eval'; EVAL_DIR.mkdir(exist_ok=True)
(EVAL_DIR / 'p4_eval.json').write_text(json.dumps(summary, indent=1), encoding='utf-8')
REPORT.to_csv(EVAL_DIR / 'test_report.csv', index=False)
zip_out = shutil.make_archive(str(Path(PROJECT_DIR) / 'p4_eval'), 'zip', EVAL_DIR)
print('saved', zip_out)

In [ ]:
# colab-only
from google.colab import files
files.download(f'{PROJECT_DIR}/p4_eval.zip')